In [483]:
import openai
# Core
import os
import re
import io
import logging
import pathlib
import sqlite3
from typing import TypedDict, List

# I/O and environment
from dotenv import load_dotenv, find_dotenv
from IPython.display import display, Markdown, Image

# LangChain
from langchain.chains import RetrievalQA, LLMChain, StuffDocumentsChain
from langchain_core.prompts import ChatPromptTemplate
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import PromptTemplate


# Pinecone
from pinecone import Pinecone
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# LangGraph
from langgraph.graph import START, StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver


In [484]:
# Config logging
logging.basicConfig(level=logging.INFO)

In [485]:
# Load enviroment
load_dotenv(find_dotenv())
memory = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False))
openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv('PINECONE_API_KEY')

# Set the api keys
openai.api_key =openai_api_key
client = openai.OpenAI()

MODEL_NAME = "gpt-4o-mini" 

In [486]:
# Inicializar OpenAIEmbeddings desde langchain
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)


pc = Pinecone(api_key=pinecone_api_key)
# Inicializar cliente Pinecone
index_name = "rag-literary-chatbot"

existing_indexes = [index.name for index in pc.list_indexes()]

if index_name in existing_indexes:
    print(f"Index '{index_name}' already exists!")
else:
    print(f"Index '{index_name}' doesn't exist. Creating new index...")
    pc.create_index(
        name=index_name,
        dimension=1536,  # adjust based on your embedding model
        metric="cosine", 
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    

# Connect to the index
index = pc.Index(index_name)
# Inicializar Pinecone usando langchain y pasando el embedding
pinecone_vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

Index 'rag-literary-chatbot' already exists!


#### Load jsonl file

In [487]:
import json

# Load your corpus list
from pathlib import Path
path_to_read = Path("outputs") 
os.makedirs(path_to_read, exist_ok=True)
json_filename = "corpus_list.json"
jsonl_filename = "corpus_list.jsonl"
file_to_read  = os.path.join(path_to_read, json_filename)
file_to_save = os.path.join(path_to_read, jsonl_filename )


with open(file_to_read, 'r', encoding='utf-8') as f:
    corpus_list = json.load(f)

# Storage for chunks
chunk_records = []

# Configuration
MAX_TOKENS = 100  # Or characters, depending on your use case

for book in corpus_list:
    book_name = book['book_name']
    author = book['author']
    sentences_tokens = book['sentences_tokens']

    # Combine sentences into chunks
    current_chunk = []
    current_token_count = 0
    chunk_index = 0

    for sentence in sentences_tokens:
        token_count = len(sentence)

        if current_token_count + token_count <= MAX_TOKENS:
            current_chunk.extend(sentence)
            current_token_count += token_count
        else:
            # Save current chunk
            chunk_records.append({
                "chunk_text": " ".join(current_chunk),
                "metadata": {
                    "book_name": book_name,
                    "author": author,
                    "chunk_index": chunk_index
                }
            })
            chunk_index += 1

            # Start new chunk
            current_chunk = sentence.copy()
            current_token_count = token_count

    # Save any remaining chunk
    if current_chunk:
        chunk_records.append({
            "chunk_text": " ".join(current_chunk),
            "metadata": {
                "book_name": book_name,
                "author": author,
                "chunk_index": chunk_index
            }
        })

print(f"Created {len(chunk_records)} chunks total.")

# Save to JSONL
with open(file_to_save, "w", encoding="utf-8") as f:
    for record in chunk_records:
        json.dump(record, f, ensure_ascii=False)
        f.write("\n")

Created 934 chunks total.


In [488]:
import os
import json

def load_json_from_folder(folder_path: str):
    """
    Load documents from a folder containing .json or .jsonl files.
    Each doc will include text and metadata.
    """
    all_docs = []

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        lower_name = filename.lower()

        try:
            if lower_name.endswith(".jsonl"):
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        record = json.loads(line)
                        text = record.get("chunk_text", "")
                        metadata = record.get("metadata", {})
                        all_docs.append({
                            "text": text,
                            "metadata": metadata
                        })
            else:
                print(f"Skipping unsupported file: {filename}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    print(f"Loaded {len(all_docs)} documents from {folder_path}")
    return all_docs


In [489]:
from pathlib import Path
path_to_read = Path("data") / "input" / "rag"
os.makedirs(path_to_read, exist_ok=True)
jsonl_file = "corpus_list.jsonl"
json_file_to_read = os.path.join(path_to_read, jsonl_file)

# Upload jsonl file and getting the metadata and chunks
if path_to_read.exists():
    print(f"Directory exists: {path_to_read.resolve()}")
    docs_in_folder = load_json_from_folder(path_to_read)
    for doc in docs_in_folder[:2]:
        print(doc["metadata"])
        print(doc["text"][:200], "...")
    
else:
    print(f"Directory does not exist: {path_to_read}")



Directory exists: F:\IA\novelBot\data\input\rag
Loaded 934 documents from data\input\rag
{'book_name': 'pride_and_prejudice', 'author': 'Jane Austen', 'chunk_index': 0}
illustration george allen publisher charing cross road london ruskin house illustration jane letter prejudice jane austen preface george saintsbury illustrations hugh thomson illustration ruskin chari ...
{'book_name': 'pride_and_prejudice', 'author': 'Jane Austen', 'chunk_index': 1}
delightful freshness humour northanger abbey completeness finish entrain obscure undoubted critical fact small scheme burlesque parody kind first rank difficulty persuasion relatively faint tone inter ...


In [490]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [491]:
# Prepare to insert vectors to Pinecone
vectors = []
for doc in docs_in_folder:
    # Generate embedding for the document text
    embedding = get_embedding(doc["text"])
    
    # Create metadata dictionary
    metadata = {
        "text": doc["text"][:100], 
        "book_name": doc["metadata"]["book_name"],
        "author": doc["metadata"]["author"],
        "chunk_index": doc["metadata"]["chunk_index"]
    }
    
    # Create a unique ID for the vector
    vector_id = f"{doc['metadata']['book_name']}_{doc['metadata']['chunk_index']}"
    
    vectors.append((vector_id, embedding, metadata))

# Upsert in batches (Pinecone recommends batches of 100)
batch_size = 16
for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i+batch_size]
    index.upsert(vectors=batch)

print(f"Successfully upserted {len(vectors)} vectors to Pinecone index {index_name}")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

Successfully upserted 934 vectors to Pinecone index rag-literary-chatbot


In [492]:
from typing_extensions import Annotated
class AgentState(TypedDict):
    task:str
    question: str
    plan:str
    context: Annotated[List[Document], "accumulate"]
    answer: str
    content: Annotated[List[str], "accumulate"]

In [493]:
model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [494]:
PLAN_PROMPT = """
You are an expert literary planner. Given a user's goal and a list of literary works, generate a clear, high-level outline for evaluating each work.

Your plan should include:
- The name of each work.
- A short summary of its main narrative characteristics, style, and themes.
- Any comparison points if multiple authors are included.
- Maximum 100 words.

If no works are provided, assume two public domain examples.

Respond in this JSON format:
{
  "plan": [
    {
      "work": "Work Title",
      "summary": "One or two sentences summarizing the work's main literary features."
    },
    ...
  ]
}
"""


In [604]:
LLM_PROMPT_LITERARY_ANALYSIS = """
You are an expert literary analyst. Your task is to generate focused analytical questions for literary texts.

INSTRUCTIONS:
1. Examine the provided context carefully
2. Look for mentions of these authors: Jane Austen, Lewis Carroll, Mark Twain, or Oscar Wilde
3. If you find one of these authors, generate 2-5 analytical questions about their work
4. If none of these authors are mentioned, still generate general literary analysis questions

AUTHOR-SPECIFIC FOCUS:
- Jane Austen: Social commentary, irony, character relationships, dialogue, marriage themes
- Lewis Carroll: Nonsense literature, wordplay, logical puzzles, fantasy elements
- Mark Twain: Satire, dialect, social criticism, humor, American themes  
- Oscar Wilde: Wit, aestheticism, paradox, social satire, moral themes

ALWAYS respond in this exact JSON format:
{
  "queries": [
    "Question 1 about the text's literary elements",
    "Question 2 about narrative technique or style",
    "Question 3 about themes or character development",
    "Question 4 about the author's distinctive approach"
  ]
}

Context to analyze: {context}

Generate specific, actionable questions that will help analyze the literary work in the context.
"""


In [ ]:
LLM_PROMPT_LITERARY_ANALYSIS_2 = """
You are an intelligent literary assistant tasked with transforming a user's goal into targeted literary analysis questions 
to help evaluate the style, narrative elements, and thematic aspects of a literary work.

Proceed only if the author's name "Jane Austen", "Lewis Carroll", "Mark Twain"  or "Oscar Wilde" is explicitly mentioned in the {context}. 
If "Jane Austen", "Lewis Carroll", "Mark Twain"  or "Oscar Wilde" is not mentioned, respond with the following JSON exactly as shown:

{{
  "error": "I cannot provide the literary analysis because Jane Austen is not the author mentioned in the text."
}}

If "Jane Austen" is present, generate 2 to 5 short, focused literary analysis questions that could help examine her writing style, 
character development, and narrative techniques.

Respond only in the following JSON format:
{{
  "queries": [
   "What narrative techniques does the author use in this work?",
   "How does the author develop the main characters and their relationships?",
   "What central themes and motifs are explored in the text?",
   "How does the dialogue reflect the historical and social context of the time?",
   "What stylistic features make this work distinctive within the author’s body of work?",
   "How does the author’s narrative style compare to that of other authors in the corpus?"
  ]
}}

Context: {context}
"""



In [606]:
from pydantic import BaseModel
from typing import List

class Queries(BaseModel):
    queries: List[str]

In [607]:
def plan_node(state: AgentState):
    works = state.get("works") or ["Pride and Prejudice by Jane Austen", "Alice's Adventures in Wonderland by Lewis Carroll", "The Adventures by Tom Sawyer", "The Picture of Dorian Gray by Oscar Wilde"]
    print(works)
    task = state.get("task") or "Create a literary analysis plan."

    # Compose full user input for clarity
    user_input = f"""
    Task: {task}

    Literary Works:
    {', '.join(works)}
    """

    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=user_input)
    ]

    response = model.invoke(messages)
    print("Raw LLM response:", response)

    return {"plan": response.content}


In [608]:
from langchain.schema import Document
from more_itertools import chunked  # helps to split lists into batches

def retriever(state: AgentState):
    documents = [
        Document(
            page_content=chunk["chunk_text"],
            metadata=chunk["metadata"]
        )
        for chunk in chunk_records
    ]

    # Create indices for Pinecone
    indices = [
        f"{chunk['metadata']['book_name']}_{chunk['metadata']['chunk_index']}"
        for chunk in chunk_records
    ]

    # Pair each doc with its ID
    doc_pairs = list(zip(documents, indices))

    # Batch size: adjust to keep each batch under 4MB — test ~50–200 docs at a time
    BATCH_SIZE = 100

    for batch in chunked(doc_pairs, BATCH_SIZE):
        batch_docs, batch_ids = zip(*batch)
        pinecone_vectorstore.add_documents(documents=list(batch_docs), ids=list(batch_ids))
        print(f"Added batch of {len(batch_docs)} chunks to Pinecone")

    print(f"Total added: {len(documents)} chunks to Pinecone vectorstore")

    # Formulate query
    query = state.get("question") or state.get("task") or "Describe the main themes in the corpus."

    # Configure retriever
    retriever = pinecone_vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': 5}
    )

    # Retrieve relevant documents
    relevant_docs = retriever.get_relevant_documents(query)
    print(f"Found {len(relevant_docs)} relevant documents for query: {query}")

    return {
        "context": relevant_docs,
        "query": query,
        "source_documents": relevant_docs
    }



In [609]:
def generate(state: AgentState):
    context_docs = state.get("context", [])
    content_chunks = state.get("content", [])

    #print("Context docs:", context_docs)
    #print("Content chunks:", content_chunks)

    # Combine available content
    if context_docs:
        docs_content = "\n\n".join(texts.page_content for texts in context_docs if hasattr(texts, "page_content"))
    elif content_chunks:
        docs_content = "\n\n".join([str(chunk) for chunk in content_chunks])
    else:
        return {"answer": "No context or content available for answer generation."}

    # Get a safe fallback question string
    question =   question = (
    (state.get('question') or '').strip()
    or (state.get('task') or '').strip()
    or "Describe the main narrative techniques, style, and themes in the literary works provided."
    )

    prompt_text = f"""
    You are an expert literary analysis assistant. Rely exclusively on the context provided to craft a clear, insightful, and well-structured answer to the user's question.

    Focus on analyzing narrative techniques, style, tone, central themes, and any distinctive features of the work or authors mentioned. If appropriate, highlight comparisons between different authors’ approaches, styles, or narrative structures.

    If the context does not contain enough information to answer the question reliably, respond exactly with: "I do not have enough information in the context to answer this question."

---

**Question:**  
{question}

---

**Context:**  
{docs_content}

---

**Answer:**
"""

    try:
        response = model.invoke([HumanMessage(content=prompt_text)])
        return {"answer": response.content}
    except Exception as e:
        print("Error during model invocation:", e)
        return {"answer": f"Error invoking model: {str(e)}"}

In [610]:
def llm_user1_node(state: AgentState):
    structured_llm = model.with_structured_output(Queries)
    messages = [
        SystemMessage(content=LLM_PROMPT_LITERARY_ANALYSIS),
        HumanMessage(content=state.get('task', ''))
    ]

    try:
        queries = structured_llm.invoke(messages)
        print("Queries:", queries)
    except Exception as e:
        return {"content": [], "error": f"Failed to generate queries: {e}"}

    if not queries or not getattr(queries, "queries", None):
        return {"content": [], "error": "No queries generated"}

    llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=pinecone_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7}),
        return_source_documents=True 
    )

    content = []
    for q in queries.queries:
        if not isinstance(q, str):
            continue
        result = qa_chain.invoke({"query": q})
        print(f"Answer: {result['result']}")
        content.append(result["result"])
       
    return {"content": [content]}

In [611]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval

In [612]:
def llm_user2_node(state: AgentState):
    structured_llm = model.with_structured_output(Queries)

    # 1) Setup retriever
    retriever = pinecone_vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 7
            #"filter": {
            #    "author": {"$in": ["Jane Austen", "Mark Twain", "Lewis Carroll", "Oscar Wilde"]}
            #}
        }
    )

    # 2) Get initial context for query planning
    query_for_context = state.get('question') or state.get('task') or "Analyze the literary style."
    retrieved_docs = retriever.get_relevant_documents(query_for_context)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs]) or "No context found."

    # 3) Generate analysis queries with the structured LLM
    prompt = LLM_PROMPT_LITERARY_ANALYSIS_2.format(context=context_text)

    messages = [
        SystemMessage(content=prompt),
        HumanMessage(content=state.get('task', ''))
    ]

    try:
        queries = structured_llm.invoke(messages)
        print("Queries:", queries)
    except Exception as e:
        return {"Evaluator2": {"content": [], "error": f"Failed to generate queries: {e}"}}

    if not queries or not getattr(queries, "queries", None):
        return {"Evaluator2": {"content": [], "error": "No queries generated"}}

    # 4) Setup your QA chain
    llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
    qa_prompt = """
    Use the following context to answer the question.
    If the answer is not in the context, say "I don't know."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={
            "prompt": PromptTemplate(
                input_variables=["context", "question"],
                template=qa_prompt
            ),
            "document_variable_name": "context"
        }
    )

    content = []
    scores = []
    reasons = []

    # 5) Loop through each query, get answer, and score it
    for q in queries.queries:
        if not isinstance(q, str):
            continue

        result = qa_chain.invoke({"query": q})
        answer = result["result"]
        content.append(answer)

        test_case = LLMTestCase(input=q, actual_output=answer)
        metric = GEval(
            name="Literary Coherence",
            criteria="The answer should be coherent, stylistically consistent, and relevant.",
            evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        )
        metric.measure(test_case)

        scores.append(metric.score)
        reasons.append(metric.reason)

    avg_score = sum(scores) / len(scores) if scores else 0.0

    return {"content": content,
        "Evaluator2": {
            "score": avg_score,
            "reason": reasons
        }
    }


In [613]:
def should_continue(state):
    return END

In [614]:
builder = StateGraph(AgentState)

In [615]:
builder.add_node("planner", plan_node)
builder.add_node("Evaluator1", llm_user1_node)
builder.add_node("Evaluator2", llm_user2_node)
builder.add_node("generate", generate)
builder.add_node("retriever", retriever)

builder.set_entry_point("planner")
builder.set_finish_point("Evaluator2")  
#builder.set_finish_point("Evaluator2")  

builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "generate")
builder.add_edge("generate", "Evaluator1")
builder.add_edge("Evaluator1", "Evaluator2")

In [616]:
builder.set_entry_point("planner")

In [617]:
builder.add_conditional_edges(
    "generate", 
    should_continue, 
    {END: END}
)

In [618]:
graph = builder.compile(checkpointer=memory)

In [619]:
from IPython.display import Image

#Image(graph.get_graph().draw_png())

In [620]:
from typing import List, Dict

# Simulación del stream y de las salidas
thread = {"configurable": {"thread_id": "1"}}

# Función para generar una salida más bonita
def print_pretty_output():
    initial_state = {
        "task": (
            "Evaluate each of the provided literary works. "
            "Summarize the main characteristics of the text, highlighting its distinctive style, themes, "
            "and narrative techniques."
        ),
        "content": []
    }

    for s in graph.stream(initial_state, thread):
        display(Markdown(f"### System answer: \n\n{s}"))

# Llamar la función
print_pretty_output()

['Pride and Prejudice by Jane Austen', "Alice's Adventures in Wonderland by Lewis Carroll", 'The Adventures by Tom Sawyer', 'The Picture of Dorian Gray by Oscar Wilde']


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Raw LLM response: content='{\n  "plan": [\n    {\n      "work": "Pride and Prejudice",\n      "summary": "Austen\'s novel features sharp social commentary and irony, focusing on themes of love, class, and reputation through the lens of Elizabeth Bennet\'s romantic entanglements."\n    },\n    {\n      "work": "Alice\'s Adventures in Wonderland",\n      "summary": "Carroll\'s whimsical narrative employs absurdity and playful language, exploring themes of identity and the nonsensical nature of reality through Alice\'s fantastical journey."\n    },\n    {\n      "work": "The Adventures of Tom Sawyer",\n      "summary": "Twain\'s coming-of-age tale combines humor and adventure, highlighting themes of childhood, freedom, and moral growth as Tom navigates life along the Mississippi River."\n    },\n    {\n      "work": "The Picture of Dorian Gray",\n      "summary": "Wilde\'s novel delves into aestheticism and moral duplicity, using a gothic style to explore themes of vanity, hedonism, and t

### System answer: 

{'planner': {'plan': '{\n  "plan": [\n    {\n      "work": "Pride and Prejudice",\n      "summary": "Austen\'s novel features sharp social commentary and irony, focusing on themes of love, class, and reputation through the lens of Elizabeth Bennet\'s romantic entanglements."\n    },\n    {\n      "work": "Alice\'s Adventures in Wonderland",\n      "summary": "Carroll\'s whimsical narrative employs absurdity and playful language, exploring themes of identity and the nonsensical nature of reality through Alice\'s fantastical journey."\n    },\n    {\n      "work": "The Adventures of Tom Sawyer",\n      "summary": "Twain\'s coming-of-age tale combines humor and adventure, highlighting themes of childhood, freedom, and moral growth as Tom navigates life along the Mississippi River."\n    },\n    {\n      "work": "The Picture of Dorian Gray",\n      "summary": "Wilde\'s novel delves into aestheticism and moral duplicity, using a gothic style to explore themes of vanity, hedonism, and the consequences of a life devoted to pleasure."\n    }\n  ]\n}'}}

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Output()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


### System answer: 

{'Evaluator2': {'content': ["I don't know.", "I don't know.", "I don't know.", "I don't know.", "I don't know.", "I don't know."]}}